In [0]:
from pyspark.sql.functions import col, current_timestamp, input_file_name

In [0]:
source_path = "s3://garvit-ipl-data-lake/raw/ipl_json"
bronze_path = "s3://garvit-ipl-data-lake/bronze/matches"
checkpoint_path = "s3://garvit-ipl-data-lake/logs/checkpoints/bronze"
schema_checkpoint_path = "s3://garvit-ipl-data-lake/logs/checkpoints/bronze_schema"

In [0]:
bronze_df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "text")
         .option("wholetext", "true")  # Reads whole JSON file as 1 record
         .option("cloudFiles.schemaLocation", schema_checkpoint_path)
         .load(source_path)
         .select(
             col("value").alias("raw_json"),
             col("_metadata.file_path").alias("source_file"),
             current_timestamp().alias("ingestion_timestamp")
         )
)
# --- 3. WRITE STREAM TO DELTA LAKE ---
query = (
    bronze_df.writeStream
             .format("delta")
             .outputMode("append")
             .option("checkpointLocation", checkpoint_path)
             .option("mergeSchema", "true")
             .trigger(availableNow=True)
             .start(bronze_path)
)
query.awaitTermination()
print("Bronze data ingestion completed successfully!")

Bronze data ingestion completed successfully!


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.bronze_ipl_matches
USING DELTA
LOCATION 's3://garvit-ipl-data-lake/bronze/matches';

In [0]:
%sql
SELECT * FROM workspace.default.bronze_ipl_matches LIMIT 5;

raw_json source_file ingestion_timestamp {
 "meta": {
 "data_version": "1.0.0",
 "created": "2020-10-20",
 "revision": 1
 },
 "info": {
 "balls_per_over": 6,
 "dates": [
 "2020-10-18"
 ],
 "event": {
 "name": "Indian Premier League",
 "match_number": 36
 },
 "gender": "male",
 "match_type": "T20",
 "officials": {
 "match_referees": [
 "J Srinath"
 ],
 "reserve_umpires": [
 "AY Dandekar"
 ],
 "tv_umpires": [
 "AK Chaudhary"
 ],
 "umpires": [
 "Nitin Menon",
 "PR Reiffel"
 ]
 },
 "outcome": {
 "result": "tie",
 "eliminator": "Kings XI Punjab"
 },
 "overs": 20,
 "player_of_match": [
 "KL Rahul"
 ],
 "players": {
 "Mumbai Indians": [
 "RG Sharma",
 "Q de Kock",
 "SA Yadav",
 "Ishan Kishan",
 "KH Pandya",
 "HH Pandya",
 "KA Pollard",
 "NM Coulter-Nile",
 "RD Chahar",
 "TA Boult",
 "JJ Bumrah"
 ],
 "Kings XI Punjab": [
 "KL Rahul",
 "MA Agarwal",
 "CH Gayle",
 "N Pooran",
 "GJ Maxwell",
 "DJ Hooda",
 "CJ Jordan",
 "M Ashwin",
 "Ravi Bishnoi",
 "Mohammed Shami",
 "Arshdeep Singh"
 ]
 },
 "registry": {
 "people": {
 "AK Chaudhary": "fdcc6236",
 "AS Roy": "e4cdf230",
 "AY Dandekar": "2efc430e",
 "Arshdeep Singh": "244048f6",
 "CH Gayle": "db584dad",
 "CJ Jordan": "ffe699c0",
 "DJ Hooda": "73ad96ed",
 "GJ Maxwell": "b681e71e",
 "HH Pandya": "dbe50b21",
 "Ishan Kishan": "752f7486",
 "J Srinath": "bad31fac",
 "JJ Bumrah": "462411b3",
 "KA Pollard": "a757b0d8",
 "KH Pandya": "5b8c830e",
 "KL Rahul": "b17e2f24",
 "M Ashwin": "e2db2409",
 "MA Agarwal": "00ea847a",
 "Mohammed Shami": "8cf9814c",
 "N Pooran": "3241e3fd",
 "NM Coulter-Nile": "56ab442f",
 "Nitin Menon": "e1d41d9e",
 "PR Reiffel": "c9354f29",
 "Q de Kock": "372455c4",
 "RD Chahar": "2ed569a0",
 "RG Sharma": "740742ef",
 "Ravi Bishnoi": "df064e1a",
 "SA Yadav": "271f83cd",
 "TA Boult": "a818c1be"
 }
 },
 "season": "2020/21",
 "team_type": "club",
 "teams": [
 "Mumbai Indians",
 "Kings XI Punjab"
 ],
 "toss": {
 "decision": "bat",
 "winner": "Mumbai Indians"
 },
 "venue": "Dubai International Cricket Stadium"
 },
 "innings": [
 {
 "team": "Mumbai Indians",
 "overs": [
 {
 "over": 0,
 "deliveries": [
 {
 "batter": "RG Sharma",
 "bowler": "GJ Maxwell",
 "non_striker": "Q de Kock",
 "runs": {
 "batter": 1,
 "extras": 0,
 "total": 1
 }
 },
 {
 "batter": "Q de Kock",
 "bowler": "GJ Maxwell",
 "non_striker": "RG Sharma",
 "runs": {
 "batter": 0,
 "extras": 0,
 "total": 0
 }
 },
 {
 "batter": "Q de Kock",
 "bowler": "GJ Maxwell",
 "non_striker": "RG Sharma",
 "runs": {
 "batter": 1,
 "extras": 0,
 "total": 1
 }
 },
 {
 "batter": "RG Sharma",
 "bowler": "GJ Maxwell",
 "non_striker": "Q de Kock",
 "runs": {
 "batter": 0,
 "extras": 0,
 "total": 0
 }
 },
 {
 "batter": "RG Sharma",
 "bowler": "GJ Maxwell",
 "non_striker": "Q de Kock",
 "runs": {
 "batter": 4,
 "extras": 0,
 "total": 4
 }
 },
 {
 "batter": "RG Sharma",
 "bowler": "GJ Maxwell",
 "non_striker": "Q de Kock",
 "runs": {
 "batter": 0,
 "extras": 0,
 "total": 0
 }
 }
 ]
 },
 {
 "over": 1,
 "deliveries": [
 {
 "batter": "Q de Kock",
 "bowler": "Mohammed Shami",
 "non_striker": "RG Sharma",
 "runs": {
 "batter": 4,
 "extras": 0,
 "total": 4
 }
 },
 {
 "batter": "Q de Kock",
 "bowler": "Mohammed Shami",
 "non_striker": "RG Sharma",
 "runs": {
 "batter": 0,
 "extras": 0,
 "total": 0
 }
 },
 {
 "batter": "Q de Kock",
 "bowler": "Mohammed Shami",
 "non_striker": "RG Sharma",
 "runs": {
 "batter": 1,
 "extras": 0,
 "total": 1
 }
 },
 {
 "batter": "RG Sharma",
 "bowler": "Mohammed Shami",
 "non_striker": "Q de Kock",
 "runs": {
 "batter": 4,
 "extras": 0,
 "total": 4
 }
 },
 {
 "batter": "RG Sharma",
 "bowler": "Mohammed Shami",
 "non_striker": "Q de Kock",
 "runs": {
 "batter": 0,
 "extras": 0,
 "total": 0
 }
 },
 {
 "batter": "RG Sharma",
 "bowler": "Mohammed Shami",
 "non_striker": "Q de Kock",
 "runs": {
 "batter": 0,
 "extras": 0,
 "total": 0
 }
 }
 ]
 },
 {
 "over": 2,
 "deliveries": [
 {
 "batter": "Q de Kock",
 "bowler": "Arshdeep Singh",
 "non_striker": "RG Sharma",
 "runs": {
 "batter": 0,
 "extras": 0,
 "total": 0
 }
 },
 {
 "bat